# 🕸️ Tutorial · Simulasi Jaringan dengan **NetworkX** saja (Google Colab)

Tutorial ini mencapai tujuan yang sama dengan tutorial Mininet + Ryu, tetapi **tanpa Mininet, Ryu, atau WSL** — cukup NetworkX. Cocok bila instalasi WSL/Mininet gagal, atau untuk belajar konsep dengan cepat di **Google Colab**.

| Bagian | Padanan di dunia Mininet/Ryu | Isi |
|---|---|---|
| 1. Galeri topologi | `mn --topo ...` | bangun single/linear/tree/ring/mesh/fat-tree/acak/JSON sebagai graf |
| 2. Metrik graf | — | derajat, diameter, konektivitas, betweenness |
| 3. Model link & jalur | `TCLink` + routing | delay antrean M/M/1, shortest-path, perbandingan algoritma |
| 4. Simulasi paket | kirim paket di Mininet | simulator *discrete-event* sederhana: RTT, throughput, antrean |
| 5. Uji ketahanan | link putus + RSTP/controller | putuskan link/node, hitung jalur cadangan |
| 6. Ekspor optimasi | topologi JSON | berkas `data/topology.json` untuk notebook optimasi |
| 7. Ekspor DDoS | flow/port stats controller | tabel aliran berlabel + fitur per jendela |

### Cara pakai
**Runtime → Run all**. Ubah parameter lewat form (slider/dropdown) di tiap sel, jalankan ulang sel itu dan sesudahnya. Hasil tersimpan di `hasil_networkx/tutorial/`.

In [ ]:
#@title 📦 Import (semua sudah ada di Colab)
import math, json, time, itertools, shutil
from collections import defaultdict, deque
from pathlib import Path
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
OUT = Path("hasil_networkx/tutorial")
OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})
print(f"networkx {nx.__version__} | numpy {np.__version__} | Colab: {IN_COLAB}")

## 1. Galeri topologi
Setiap topologi dibuat sebagai graf **switch + host**. Switch diberi atribut posisi; link diberi `delay` (ms), `bandwidth` (Mbps), dan `loss` (probabilitas). Ini cetak biru yang sama seperti pada tutorial Mininet, sehingga dataset yang diekspor kompatibel.

In [ ]:
#@title ⚙️ Parameter topologi
JENIS = "tree"  #@param ["single", "linear", "tree", "ring", "mesh", "fat-tree", "acak", "json"]
JUMLAH_NODE = 6  #@param {type:"slider", min:2, max:40, step:1}
TREE_DEPTH = 2  #@param {type:"slider", min:1, max:4, step:1}
TREE_FANOUT = 2  #@param {type:"slider", min:2, max:4, step:1}
FATTREE_K = 4  #@param {type:"slider", min:2, max:6, step:2}
SEED = 42  #@param {type:"integer"}

In [ ]:
def rand_attrs(rng):
    return dict(delay=float(rng.choice([1, 2, 3, 5])), bandwidth=float(rng.choice([20, 50, 100])),
                loss=round(float(rng.choice([0, 0, 0.001, 0.002])), 4))

def make_topology(kind, n=6, depth=2, fanout=2, k=4, seed=42):
    rng = np.random.default_rng(seed)
    S = nx.Graph()
    if kind == "single":
        S.add_node("s1")
    elif kind == "linear":
        nx.add_path(S, [f"s{i+1}" for i in range(n)])
    elif kind == "tree":
        S = nx.relabel_nodes(nx.balanced_tree(fanout, depth), lambda u: f"s{u+1}")
    elif kind == "ring":
        nx.add_cycle(S, [f"s{i+1}" for i in range(n)])
    elif kind == "mesh":
        S = nx.relabel_nodes(nx.complete_graph(n), lambda u: f"s{u+1}")
    elif kind == "fat-tree":
        core = [f"s{i+1}" for i in range((k // 2) ** 2)]
        S.add_nodes_from(core); cnt = len(core)
        for _ in range(k):
            aggs = [f"s{cnt+i+1}" for i in range(k // 2)]; cnt += k // 2
            edges = [f"s{cnt+i+1}" for i in range(k // 2)]; cnt += k // 2
            for i, a in enumerate(aggs):
                for j in range(k // 2):
                    S.add_edge(a, core[i * (k // 2) + j])
                for e in edges:
                    S.add_edge(a, e)
            for e in edges:
                S.nodes[e]["leaf"] = True
    elif kind == "acak":
        pos = {f"s{i+1}": (rng.uniform(0, 1000), rng.uniform(0, 1000)) for i in range(n)}
        S.add_nodes_from(pos)
        for u in pos:
            for v in sorted((x for x in pos if x != u), key=lambda x: math.dist(pos[u], pos[x]))[:3]:
                S.add_edge(u, v)
        while not nx.is_connected(S):
            cc = list(nx.connected_components(S))
            S.add_edge(min(cc[0]), min(cc[1]))
    elif kind == "json":
        data = json.loads(Path("data/topology.json").read_text()) if Path("data/topology.json").exists() else \
               {"nodes": [f"S{i+1}" for i in range(6)], "edges": [{"source": f"S{i+1}", "target": f"S{i+2}", "delay": 2, "bandwidth": 100, "loss": 0.001} for i in range(5)]}
        name = {u: f"s{i+1}" for i, u in enumerate(data["nodes"])}
        for e in data["edges"]:
            S.add_edge(name[e["source"]], name[e["target"]], delay=float(e["delay"]), bandwidth=float(e["bandwidth"]), loss=float(e["loss"]))
    G = nx.Graph(kind=kind)
    for u in S:
        G.add_node(u, type="switch", leaf=S.nodes[u].get("leaf", False))
    for u, v, d in S.edges(data=True):
        G.add_edge(u, v, **(d if d else rand_attrs(rng)))
    # host
    if kind == "single":
        attach = {"s1": n}
    elif kind == "tree":
        attach = {u: fanout for u in S if S.degree(u) == 1}
    elif kind == "fat-tree":
        attach = {u: k // 2 for u in S if S.nodes[u].get("leaf")}
    else:
        attach = {u: 1 for u in S}
    h = 0
    for sw in sorted(attach, key=lambda s: int(s[1:])):
        for _ in range(attach[sw]):
            h += 1
            G.add_node(f"h{h}", type="host", ip=f"10.0.0.{h}", mac=f"00:00:00:00:00:{h:02x}")
            G.add_edge(f"h{h}", sw, delay=0.0, bandwidth=1000.0, loss=0.0)
    return G

def sw(G):
    return sorted((u for u in G if G.nodes[u]["type"] == "switch"), key=lambda s: int(s[1:]))
def hs(G):
    return sorted((u for u in G if G.nodes[u]["type"] == "host"), key=lambda s: int(s[1:]))

def layout(G):
    S = G.subgraph(sw(G))
    if G.graph["kind"] in ("tree", "fat-tree") and S.number_of_nodes() > 1:
        roots = sw(G)[:1] if G.graph["kind"] == "tree" else sw(G)[:(FATTREE_K // 2) ** 2 or 1]
        lv = nx.multi_source_dijkstra_path_length(S, roots)
        layers = defaultdict(list)
        for u, l in lv.items():
            layers[l].append(u)
        pos = {u: (i - (len(us) - 1) / 2, -l) for l, us in layers.items() for i, u in enumerate(sorted(us))}
    elif G.graph["kind"] in ("ring", "mesh"):
        pos = nx.circular_layout(S)
    elif "pos" in next(iter(S.nodes(data=True)))[1] if S.number_of_nodes() else False:
        pos = {u: S.nodes[u]["pos"] for u in S}
    else:
        pos = nx.kamada_kawai_layout(S) if S.number_of_edges() else {sw(G)[0]: (0, 0)}
    rng = np.random.default_rng(1)
    for h in hs(G):
        base = np.array(pos[next(G.neighbors(h))], float)
        pos[h] = tuple(base + np.array([0, -0.35]) + rng.uniform(-0.2, 0.2, 2))
    return pos

def draw(G, ax, title=None, hi_edges=None, hi_nodes=None):
    pos = layout(G)
    nx.draw_networkx_edges(G, pos, ax=ax, edge_color="#c3cbd4")
    if hi_edges:
        nx.draw_networkx_edges(G, pos, ax=ax, edgelist=hi_edges, edge_color="#e8743b", width=3)
    nx.draw_networkx_nodes(G, pos, nodelist=sw(G), ax=ax, node_color="#2f6db5", node_shape="s", node_size=200)
    nx.draw_networkx_nodes(G, pos, nodelist=hs(G), ax=ax, node_color="#1baf7a", node_size=90)
    if hi_nodes:
        nx.draw_networkx_nodes(G, pos, nodelist=hi_nodes, ax=ax, node_color="black", node_size=140)
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=6, font_color="white")
    ax.set_title(title or G.graph["kind"], fontsize=10); ax.axis("off")
    return pos

G = make_topology(JENIS, n=JUMLAH_NODE, depth=TREE_DEPTH, fanout=TREE_FANOUT, k=FATTREE_K, seed=SEED)
fig, ax = plt.subplots(figsize=(7, 5)); draw(G, ax, f"{JENIS}: {len(sw(G))} switch, {len(hs(G))} host")
plt.tight_layout(); plt.savefig(OUT / "topologi_aktif.png", dpi=150); plt.show()

# galeri semua jenis
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
gallery = [("single", {}), ("linear", dict(n=5)), ("tree", dict(depth=2, fanout=2)), ("ring", dict(n=6)),
           ("mesh", dict(n=5)), ("fat-tree", dict(k=4)), ("acak", dict(n=10)), ("json", {})]
for ax, (kind, kw) in zip(axes.ravel(), gallery):
    Gk = make_topology(kind, seed=SEED, **kw)
    S = Gk.subgraph(sw(Gk))
    loop = "ber-loop" if S.number_of_nodes() > 1 and not nx.is_forest(S) else "tanpa loop"
    draw(Gk, ax, f"{kind} · {loop}")
plt.tight_layout(); plt.savefig(OUT / "galeri_topologi.png", dpi=150); plt.show()

## 2. Metrik graf
NetworkX menghitung sifat struktural topologi yang menjelaskan perilaku di tutorial Mininet: diameter (RTT maksimum antar switch), konektivitas link/node (ketahanan terhadap link putus), dan betweenness (switch yang paling sering dilalui → calon bottleneck).

In [ ]:
rows = []
for kind, kw in gallery:
    Gk = make_topology(kind, seed=SEED, **kw)
    S = Gk.subgraph(sw(Gk))
    if S.number_of_nodes() < 2:
        continue
    deg = [d for _, d in S.degree()]
    rows.append(dict(topologi=kind, switch=S.number_of_nodes(), link=S.number_of_edges(),
                     derajat_rata=np.mean(deg), diameter=nx.diameter(S),
                     konektivitas_link=nx.edge_connectivity(S), konektivitas_node=nx.node_connectivity(S),
                     jalur_alternatif="ya" if not nx.is_forest(S) else "tidak"))
metrik = pd.DataFrame(rows).set_index("topologi")
metrik.to_csv(OUT / "metrik_graf.csv")
display(metrik.round(2))

S = G.subgraph(sw(G))
if S.number_of_edges():
    bc = nx.betweenness_centrality(S)
    fig, ax = plt.subplots(figsize=(7, 5)); pos = layout(G)
    nx.draw_networkx_edges(S, pos, ax=ax, edge_color="#c3cbd4")
    nodes = nx.draw_networkx_nodes(S, pos, ax=ax, node_color=[bc[u] for u in S], cmap="YlOrRd", node_size=600)
    nx.draw_networkx_labels(S, pos, ax=ax, font_size=7)
    plt.colorbar(nodes, ax=ax, label="betweenness (semakin tinggi = calon bottleneck)")
    ax.set_title(f"Betweenness centrality — {JENIS}"); ax.axis("off")
    plt.tight_layout(); plt.savefig(OUT / "betweenness.png", dpi=150); plt.show()

## 3. Model link & perbandingan algoritma routing
Delay link memakai model antrean **M/M/1** (seperti proyek reinforcement_and_federated_network): pada utilisasi ρ, `delay = delay_dasar + 8·(12/bandwidth)·ρ/(1−ρ) + loss·200ms`. Dengan ρ ini kita bandingkan 3 algoritma jalur: **Hop-Count**, **Dijkstra (delay)**, dan **Widest-Path** (bandwidth bottleneck terbesar).

In [ ]:
#@title ⚙️ Beban jaringan
UTILISASI_RATA = 0.5  #@param {type:"slider", min:0.05, max:0.95, step:0.05}
rng = np.random.default_rng(SEED + 3)
for u, v, d in G.edges(data=True):
    if G.nodes[u]["type"] == "switch" and G.nodes[v]["type"] == "switch":
        rho = float(np.clip(UTILISASI_RATA + rng.normal(0, 0.15), 0.05, 0.95))
        d["util"] = rho
        d["delay_aktual"] = d["delay"] + 8 * (12 / d["bandwidth"]) * rho / (1 - rho) + d["loss"] * 200
    else:
        d["util"] = 0.0; d["delay_aktual"] = d["delay"]

def pedges(p):
    return list(zip(p[:-1], p[1:]))
def widest(s, t):
    for thr in sorted({G[u][v]["bandwidth"] for u, v in G.edges()}, reverse=True):
        H = G.edge_subgraph([(u, v) for u, v in G.edges() if G[u][v]["bandwidth"] >= thr])
        if s in H and t in H and nx.has_path(H, s, t):
            return nx.dijkstra_path(H, s, t, weight="delay_aktual")
ALG = {"Hop-Count": lambda s, t: nx.shortest_path(G, s, t),
       "Dijkstra (delay)": lambda s, t: nx.dijkstra_path(G, s, t, weight="delay_aktual"),
       "Widest-Path": widest}
src, dst = hs(G)[0], hs(G)[-1]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, fn) in zip(axes, ALG.items()):
    p = fn(src, dst)
    d = sum(G[u][v]["delay_aktual"] for u, v in pedges(p))
    bw = min(G[u][v]["bandwidth"] for u, v in pedges(p))
    draw(G, ax, f"{name}\n{len(pedges(p))} hop · {d:.1f} ms · bott {bw:g}M", hi_edges=pedges(p), hi_nodes=[src, dst])
plt.tight_layout(); plt.savefig(OUT / "routing.png", dpi=150); plt.show()

## 4. Simulasi paket *discrete-event*
Simulator ringkas: tiap link punya antrean dengan laju layan sebanding bandwidth. Kita kirim aliran paket antar host sepanjang jalur yang dipilih, dan ukur **RTT**, **throughput**, serta **antrean** — padanan `ping`/`iperf` di Mininet, tanpa Mininet.

In [ ]:
#@title ⚙️ Parameter simulasi paket
LAJU_KIRIM_PPS = 400  #@param {type:"slider", min:50, max:3000, step:50}
DURASI_SIM_S = 5  #@param {type:"slider", min:1, max:20, step:1}
UKURAN_PAKET_BYTE = 1000  #@param {type:"slider", min:200, max:1500, step:100}

def simulate_flow(G, path, pps, dur, size, seed=0):
    """Antrean M/M/1 per link di sepanjang jalur; kembalikan RTT per paket & statistik."""
    rng = np.random.default_rng(seed)
    edges = pedges(path)
    serv = {e: size * 8 / (G[e[0]][e[1]]["bandwidth"] * 1e6) for e in edges}   # waktu layan (s/paket)
    free_at = {e: 0.0 for e in edges}
    prop = {e: G[e[0]][e[1]]["delay"] / 1000 for e in edges}
    loss_p = {e: G[e[0]][e[1]]["loss"] for e in edges}
    n = int(pps * dur)
    arr = np.cumsum(rng.exponential(1 / pps, n))
    rtts, dropped = [], 0
    for t0 in arr:
        t = t0; ok = True
        for e in edges:
            if rng.random() < loss_p[e]:
                ok = False; break
            start = max(t, free_at[e])
            free_at[e] = start + serv[e]
            t = start + serv[e] + prop[e]
        if ok:
            rtts.append((t - t0) * 2 * 1000)      # ms, pulang-pergi
        else:
            dropped += 1
    rtts = np.array(rtts)
    delivered = len(rtts)
    return dict(rtt_rata_ms=rtts.mean() if delivered else np.nan, rtt_p95_ms=np.percentile(rtts, 95) if delivered else np.nan,
                throughput_mbps=delivered * size * 8 / dur / 1e6, loss_pct=100 * dropped / n,
                antrean_maks_ms=1000 * max((serv[e] for e in edges)) * max(1, int(pps * max(serv.values()))), rtts=rtts)

path = nx.dijkstra_path(G, src, dst, weight="delay_aktual")
res = {}
for pps in [LAJU_KIRIM_PPS // 2, LAJU_KIRIM_PPS, LAJU_KIRIM_PPS * 2]:
    res[pps] = simulate_flow(G, path, pps, DURASI_SIM_S, UKURAN_PAKET_BYTE, seed=SEED)
tbl = pd.DataFrame({f"{k} pps": {kk: vv for kk, vv in v.items() if kk != "rtts"} for k, v in res.items()}).T
display(tbl.round(3))
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for pps, r in res.items():
    if len(r["rtts"]):
        axes[0].hist(r["rtts"], bins=40, alpha=0.5, label=f"{pps} pps")
axes[0].set_xlabel("RTT (ms)"); axes[0].set_ylabel("jumlah paket"); axes[0].legend(fontsize=8); axes[0].set_title(f"Distribusi RTT {src}→{dst}")
axes[1].plot(list(res), [r["throughput_mbps"] for r in res.values()], "-o", color="#2f6db5", label="throughput")
axes[1].axhline(min(G[u][v]["bandwidth"] for u, v in pedges(path)), color="red", ls="--", label="bandwidth bottleneck")
axes[1].set_xlabel("laju kirim (pps)"); axes[1].set_ylabel("Mbps"); axes[1].legend(fontsize=8); axes[1].set_title("Throughput vs beban")
plt.tight_layout(); plt.savefig(OUT / "simulasi_paket.png", dpi=150); plt.show()

## 5. Uji ketahanan: putus link / node
Padanan uji "link putus" di Mininet, dihitung analitik: untuk tiap link (atau node) kita hapus dari graf, lalu cek apakah pasangan host masih terhubung dan berapa delay jalur cadangannya. Topologi ber-loop (ring/mesh/fat-tree) punya jalur cadangan; topologi pohon tidak.

In [ ]:
S = G.subgraph(sw(G)).copy()
base = {(a, b): nx.dijkstra_path_length(G, a, b, weight="delay_aktual") for a, b in itertools.combinations(hs(G), 2)}
rows = []
for e in list(S.edges()):
    H = G.copy(); H.remove_edge(*e)
    putus = sum(1 for a, b in base if not nx.has_path(H, a, b))
    naik = [nx.dijkstra_path_length(H, a, b, weight="delay_aktual") - base[(a, b)]
            for a, b in base if nx.has_path(H, a, b)]
    rows.append(dict(link=f"{e[0]}-{e[1]}", pasangan_terputus=putus, kenaikan_delay_rata_ms=np.mean(naik) if naik else 0))
ket = pd.DataFrame(rows).sort_values("pasangan_terputus", ascending=False)
ket.to_csv(OUT / "ketahanan_link.csv", index=False)
print(f"Rata-rata pasangan terputus bila satu link gagal: {ket.pasangan_terputus.mean():.2f}")
display(ket.head(8))
fig, ax = plt.subplots(figsize=(7, 5)); pos = layout(G)
crit = dict(zip(ket.link, ket.pasangan_terputus))
ec = [crit.get(f"{u}-{v}", crit.get(f"{v}-{u}", 0)) for u, v in S.edges()]
nx.draw_networkx_edges(S, pos, ax=ax, edge_color=ec, edge_cmap=plt.cm.Reds, width=3, edge_vmin=0)
nx.draw_networkx_nodes(S, pos, ax=ax, node_color="#2f6db5", node_size=500)
nx.draw_networkx_labels(S, pos, ax=ax, font_size=7, font_color="white")
ax.set_title("Link kritis (merah = paling banyak memutus pasangan bila gagal)"); ax.axis("off")
plt.tight_layout(); plt.savefig(OUT / "link_kritis.png", dpi=150); plt.show()

## 6. Ekspor dataset untuk **optimasi jaringan**
Topologi aktif diekspor ke JSON berformat `data/topology.json`, lengkap dengan `hosts` dan `pos`, sehingga dapat dimuat `01_optimasi_jaringan.ipynb` dan notebook optimasi Colab.

In [ ]:
def export_json(G, path, description):
    S = G.subgraph(sw(G)); pos = layout(G)
    data = {"description": description, "nodes": sw(G),
            "edges": [dict(source=u, target=v, delay=G[u][v]["delay"], bandwidth=G[u][v]["bandwidth"], loss=G[u][v]["loss"]) for u, v in S.edges()],
            "hosts": {h: next(G.neighbors(h)) for h in hs(G)},
            "pos": {u: [round(float(pos[u][0]), 3), round(float(pos[u][1]), 3)] for u in sw(G)}}
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(data, indent=1))
    return path
p = export_json(G, OUT / f"topologi_{JENIS}.json", f"Topologi {JENIS} dari tutorial NetworkX")
print("ekspor:", p)
print("Muat di notebook optimasi:  TOPOLOGY_SOURCE='file'; path =", p.name)

## 7. Ekspor dataset untuk **deteksi DDoS**
Simulator aliran menghasilkan tabel **flow** berlabel (normal vs serangan) dengan skema yang sama seperti keluaran controller: tiap baris adalah satu aliran menuju server dengan `packet_count`, `byte_count`, IP sumber. Fase serangan memakai banyak IP sumber (meniru spoofing) sehingga entropi IP sumber naik. Model deteksi lengkap ada di `02_deteksi_ddos_networkx_colab.ipynb`; di sini kita hanya membentuk dataset + fitur per jendela.

In [ ]:
#@title ⚙️ Parameter dataset DDoS
SERVER_IP = "10.0.0.100"  #@param {type:"string"}
LAJU_SERANGAN_PPS = 300  #@param {type:"slider", min:20, max:2000, step:20}
JUMLAH_IP_PALSU = 150  #@param {type:"slider", min:1, max:500, step:1}
DETIK_PER_FASE = 10  #@param {type:"slider", min:4, max:60, step:2}

rng = np.random.default_rng(SEED + 9)
clients = [h for h in hs(G)]
scenario = [("normal", DETIK_PER_FASE), ("serangan", DETIK_PER_FASE), ("normal", DETIK_PER_FASE)]
flow_rows, feat_rows, t = [], [], 0.0
for phase, dur in scenario:
    for _ in range(dur):
        t += 1.0
        srcs = defaultdict(lambda: [0, 0])
        for h in clients:                                  # trafik normal
            n = rng.poisson(30)
            srcs[G.nodes[h]["ip"]][0] += n; srcs[G.nodes[h]["ip"]][1] += n * int(rng.normal(650, 120))
        if phase == "serangan":                            # flood dari banyak IP palsu
            total = int(LAJU_SERANGAN_PPS)
            parts = rng.multinomial(total, [1 / JUMLAH_IP_PALSU] * JUMLAH_IP_PALSU)
            for i, c in enumerate(parts):
                if c:
                    ip = f"172.16.{i // 254 % 254}.{i % 254 + 1}"
                    srcs[ip][0] += int(c); srcs[ip][1] += int(c) * 60
        for ip, (pk, by) in srcs.items():
            flow_rows.append(dict(ts=round(t, 1), src=ip, dst=SERVER_IP, packet_count=pk, byte_count=by,
                                  label=0 if phase == "normal" else 1))
        pks = np.array([v[0] for v in srcs.values()]); p = pks / pks.sum()
        feat_rows.append(dict(ts=round(t, 1), phase=phase, pkt_rate=pks.sum(), n_src_ip=len(srcs),
                              src_entropy=float(-(p * np.log2(p)).sum()), label=0 if phase == "normal" else 1))
flows = pd.DataFrame(flow_rows); feats = pd.DataFrame(feat_rows)
flows.to_csv(OUT / "ddos_flows.csv", index=False); feats.to_csv(OUT / "ddos_fitur.csv", index=False)
print(f"{len(flows)} baris flow, {len(feats)} jendela fitur ({int(feats.label.sum())} jendela serangan)")
display(feats.groupby("phase")[["pkt_rate", "n_src_ip", "src_entropy"]].mean().round(2))

fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
for ax, (col, lbl) in zip(axes, [("pkt_rate", "paket/s ke server"), ("src_entropy", "entropi IP sumber (bit)")]):
    ax.plot(feats.ts, feats[col], color="#2f6db5")
    for _, r in feats[feats.label == 1].iterrows():
        ax.axvspan(r.ts - 0.5, r.ts + 0.5, color="#e8743b", alpha=0.08)
    ax.set_xlabel("waktu (s)"); ax.set_title(lbl, fontsize=10)
plt.tight_layout(); plt.savefig(OUT / "ddos_timeline.png", dpi=150); plt.show()

## 8. Unduh hasil

In [ ]:
#@title 💾 Unduh ZIP
UNDUH_ZIP = False  #@param {type:"boolean"}
for f in sorted(OUT.iterdir()):
    print(f"  {f}  ({f.stat().st_size/1024:.1f} KB)")
if UNDUH_ZIP:
    z = shutil.make_archive("hasil_tutorial_networkx", "zip", OUT)
    if IN_COLAB:
        from google.colab import files
        files.download(z)
    else:
        print("ZIP:", z)

### Lanjutan
- **Optimasi**: muat `topologi_<jenis>.json` ke `01_optimasi_jaringan_networkx_colab.ipynb` (opsi file JSON), bandingkan RL & federated Q-routing pada topologi berbeda.
- **Deteksi DDoS**: `ddos_flows.csv` memakai skema yang sama dengan `02_deteksi_ddos_networkx_colab.ipynb`; latih DT/RF/KNN/SVM di sana.
- Ubah `JENIS` ke fat-tree lalu amati di bagian 5 bahwa tidak ada link yang memutus pasangan (banyak jalur cadangan).